# Decode and Evaluate Without Moving the Goalposts

Specify stable temperature, nucleus boundaries, EOS behavior, and honest metrics.

## Before You Start

Course 10. Entry course: 1. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** scratch-06_lora.

## What You Will Be Able to Do

- Implement stable top-k and nucleus filtering
- Aggregate perplexity correctly
- Separate model quality, decoding policy, and test leakage

## Sampling is a policy over logits

For positive temperature tau, softmax uses logits divided by tau. Compute finite centered logit differences before dividing so large common offsets do not erase small gaps. Only when a centered difference overflows and tau>=1, recover that entry by dividing the logits before subtracting. Smaller temperatures can saturate lower scores to zero probability. A smaller temperature sharpens the distribution; it does not add knowledge. Our decoding_probs applies top-k first (if k is provided), renormalizes, then nucleus filtering. Sort by descending probability, breaking ties by smaller token ID. Keep the smallest prefix whose cumulative mass reaches p, including the crossing token, then renormalize. Require 0<p<=1 and positive temperature.

This order is part of the API. Applying top-p before top-k can yield a different distribution. Greedy decoding is a separate argmax mode, not a division by temperature zero. The generation loop samples using a seeded generator, appends each chosen token, and stops at EOS or a maximum new-token budget. Context cropping must match the architecture's positional assumptions.

## A scoreboard needs a fixed protocol

Model likelihood uses teacher-forced logits before sampling filters. Do not report top-k-renormalized likelihood as the model's perplexity: it changes the distribution and can assign zero probability to legitimate targets. Aggregate NLL sums over valid targets. Compare per-token perplexity only when tokenization and scoring boundaries match; cross-tokenizer comparisons need a common content measure such as bytes with a clearly specified representation.

Task metrics use generated answers. Fix prompts, decoding parameters, seeds, stopping rules, normalization, and verifier before evaluating. Select changes on validation, not the final test set. Report failures as well as successes, and confidence intervals when sample sizes justify them. The tiny capstone demonstrates this discipline, not a public benchmark score or a production quality claim.

## Worked Example: The nucleus includes the crossing token

Sorted probabilities are [0.5,0.3,0.15,0.05], and p=0.7.

1. **Accumulate:** After token 0 the mass is 0.5, below 0.7. After token 1 it is 0.8, enough.
2. **Retain:** Keep tokens 0 and 1, not just token 0.
3. **Renormalize:** The retained distribution is [0.625,0.375,0,0].

Boundary and tie rules matter for reproducibility, especially on small examples.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Explicit top-k then nucleus sampling distribution."""
import numpy as np

def decoding_probs(logits, temperature=1.0, top_k=None, top_p=1.0):
    """Return probabilities in original vocabulary order from finite 1D logits.

Nonempty logits; temperature>0; 0<top_p<=1; top_k None or integer 1..V.
Apply temperature, top-k, normalize, then top-p and normalize again.
Descending ties prefer smaller IDs. Include the nucleus crossing token.
Reject invalid arguments with ValueError; do not mutate logits.
Scores below the float64 range and exponential underflow yield zero probability."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

A dataset has total natural-log NLL 20 over 10 scored tokens.

- Mean NLL (not perplexity)

<details><summary>Worked answers</summary>

- 2: 20/10=2. Perplexity is exp(2), about 7.389.

</details>

## Experiment

Run `experiments/scratch_training.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- Record generated samples at fixed seeds alongside held-out NLL; neither alone is a complete quality assessment.
- Use extreme logits and tied probabilities as numerical tests, not just realistic-looking sentences.

## Transfer Problems

1. Compare top-k then top-p with the reversed order and find a counterexample.
2. Add a maximum-token stopping reason distinct from EOS termination to the generation report.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.


## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Explicit top-k then nucleus sampling distribution."""
import numpy as np

def decoding_probs(logits, temperature=1.0, top_k=None, top_p=1.0):
    """Return probabilities in original vocabulary order from finite 1D logits.

Nonempty logits; temperature>0; 0<top_p<=1; top_k None or integer 1..V.
Apply temperature, top-k, normalize, then top-p and normalize again.
Descending ties prefer smaller IDs. Include the nucleus crossing token.
Reject invalid arguments with ValueError; do not mutate logits.
Scores below the float64 range and exponential underflow yield zero probability."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import math
import numpy as np


def _decoding_oracle(logits, temperature, top_k, top_p):
    order = sorted(range(len(logits)), key=lambda index: (-float(logits[index]), index))
    if top_k is not None:
        order = order[:top_k]
    weights = [math.exp((float(logits[index])-float(logits[order[0]]))/temperature) for index in order]
    probabilities = [weight/sum(weights) for weight in weights]
    selected, mass = [], 0.
    for index, probability in zip(order, probabilities):
        selected.append((index, probability))
        mass += probability
        if mass >= top_p:
            break
    result = np.zeros(len(logits))
    for index, probability in selected:
        result[index] = probability/sum(value for _, value in selected)
    return result


z = np.log([.5, .3, .15, .05])
assert np.allclose(decoding_probs(z, top_p=.7), [.625, .375, 0, 0]), 'Include the crossing token'
assert np.allclose(decoding_probs(z, top_k=2, top_p=.6), [1, 0, 0, 0]), 'Top-k renormalization precedes top-p'
assert np.allclose(decoding_probs([0, 0, 0], top_k=2), [.5, .5, 0]), 'Stable ID tie-breaking'
assert np.allclose(decoding_probs([10000, 10001]), [1/(1+np.e), np.e/(1+np.e)])
assert np.allclose(decoding_probs(np.log([.2, .8]), temperature=2), [1/3, 2/3])
assert np.allclose(decoding_probs(z, top_p=1), np.exp(z))
assert np.allclose(z, np.log([.5, .3, .15, .05]))
huge = np.array([1e308, -1e308])
before = huge.copy()
with np.errstate(over='raise', under='raise', invalid='raise'):
    expected = np.array([1., np.exp(-2.)])/(1+np.exp(-2.))
    np.testing.assert_allclose(decoding_probs(huge, temperature=1e308, top_p=.9), expected, rtol=1e-14, atol=0.)
    np.testing.assert_allclose(decoding_probs([1e308, 1e308], temperature=.1), [.5, .5], rtol=0., atol=0.)
    np.testing.assert_allclose(decoding_probs([1e308, np.nextafter(1e308, 0.)], temperature=.5), [1., 0.], rtol=0., atol=0.)
    np.testing.assert_allclose(decoding_probs(huge, temperature=.1, top_p=.9), [1., 0.], rtol=0., atol=0.)
    np.testing.assert_allclose(decoding_probs([1e308, 1e308, 1e308], temperature=.25, top_k=2, top_p=.6), [.5, .5, 0.], rtol=0., atol=0.)
    np.testing.assert_allclose(decoding_probs([1e308, 1e308, 1e308], temperature=.25, top_k=2, top_p=.4), [1., 0., 0.], rtol=0., atol=0.)
assert np.array_equal(huge, before)
offset_logits = np.array([2**54, 2**54-2], dtype=float)
before = offset_logits.copy()
expected = np.array([1., np.exp(-2/3)])/(1+np.exp(-2/3))
np.testing.assert_allclose(decoding_probs(offset_logits, temperature=3., top_p=1.), expected, rtol=1e-14, atol=0.)
np.testing.assert_allclose(decoding_probs(offset_logits, temperature=3., top_p=.7), expected, rtol=1e-14, atol=0.)
np.testing.assert_allclose(decoding_probs(offset_logits, temperature=3., top_p=.6), [1., 0.], rtol=0., atol=0.)
assert np.array_equal(offset_logits, before)
base_logits = np.array([-2., 0., -2., -4.])
offset_logits = base_logits + 2**54
before = offset_logits.copy()
for top_k in [None, 1, 2, 3]:
    for top_p in [.6, .7, 1.]:
        expected = _decoding_oracle(base_logits, 3., top_k, top_p)
        np.testing.assert_allclose(decoding_probs(offset_logits, 3., top_k, top_p), expected, rtol=1e-14, atol=0.)
        np.testing.assert_allclose(decoding_probs(offset_logits, 3., top_k, top_p), decoding_probs(base_logits, 3., top_k, top_p), rtol=1e-14, atol=0.)
expected = np.array([np.exp(-2/3), 1., 0., 0.])/(1+np.exp(-2/3))
np.testing.assert_allclose(decoding_probs(offset_logits, temperature=3., top_k=2, top_p=.7), expected, rtol=1e-14, atol=0.)
np.testing.assert_allclose(decoding_probs(offset_logits, temperature=3., top_k=2, top_p=.6), [0., 1., 0., 0.], rtol=0., atol=0.)
assert np.array_equal(offset_logits, before)
rng = np.random.default_rng(717)
for logits in [rng.normal(size=7)*3, np.array([2., 2., -1., 0., 2., -4., .5])]:
    before = logits.copy()
    for temperature in [.1, .8, 1., 3.]:
        for top_k in [None, 1, 3]:
            for top_p in [.4, .9, 1.]:
                expected = _decoding_oracle(logits, temperature, top_k, top_p)
                np.testing.assert_allclose(decoding_probs(logits, temperature, top_k, top_p), expected, rtol=1e-13, atol=1e-15)
    assert np.array_equal(logits, before)
for kwargs in [dict(temperature=0), dict(top_p=0), dict(top_p=1.1), dict(top_k=0), dict(top_k=True), dict(top_k=5)]:
    try:
        decoding_probs(z, **kwargs)
    except ValueError:
        pass
    else:
        raise AssertionError('Reject invalid decoding controls')
print('Decoding-policy checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Explicit top-k then nucleus sampling distribution."""
import numpy as np


def decoding_probs(logits, temperature=1., top_k=None, top_p=1.):
    """Return probabilities in original vocabulary order from finite 1D logits.

    Nonempty logits; temperature>0; 0<top_p<=1; top_k None or integer 1..V.
    Apply temperature, top-k, normalize, then top-p and normalize again.
    Descending ties prefer smaller IDs. Include the nucleus crossing token.
    Reject invalid arguments with ValueError; do not mutate logits.
    Scores below the float64 range and exponential underflow yield zero probability.
    """
    z = np.asarray(logits, float)
    if z.ndim != 1 or not z.size or not np.isfinite(z).all() or not np.isfinite(temperature) or temperature <= 0 or not np.isfinite(top_p) or not 0 < top_p <= 1:
        raise ValueError('invalid logits or sampling parameters')
    if top_k is not None and (not isinstance(top_k, int) or isinstance(top_k, bool) or not 1 <= top_k <= len(z)):
        raise ValueError('invalid top_k')
    order = np.argsort(-z, kind='stable')
    if top_k is not None:
        order = order[:top_k]
    with np.errstate(over='ignore', under='ignore'):
        centered = z[order] - z[order[0]]
        scores = centered / temperature
        if temperature >= 1:
            overflowed = np.isneginf(centered)
            scores[overflowed] = z[order[overflowed]] / temperature - z[order[0]] / temperature
        probs = np.exp(scores)
        probs /= probs.sum()
        count = min(len(probs), int(np.searchsorted(np.cumsum(probs), top_p, side='left')) + 1)
        result = np.zeros_like(z)
        result[order[:count]] = probs[:count] / probs[:count].sum()
    return result


In [ ]:
import math
import numpy as np


def _decoding_oracle(logits, temperature, top_k, top_p):
    order = sorted(range(len(logits)), key=lambda index: (-float(logits[index]), index))
    if top_k is not None:
        order = order[:top_k]
    weights = [math.exp((float(logits[index])-float(logits[order[0]]))/temperature) for index in order]
    probabilities = [weight/sum(weights) for weight in weights]
    selected, mass = [], 0.
    for index, probability in zip(order, probabilities):
        selected.append((index, probability))
        mass += probability
        if mass >= top_p:
            break
    result = np.zeros(len(logits))
    for index, probability in selected:
        result[index] = probability/sum(value for _, value in selected)
    return result


z = np.log([.5, .3, .15, .05])
assert np.allclose(decoding_probs(z, top_p=.7), [.625, .375, 0, 0]), 'Include the crossing token'
assert np.allclose(decoding_probs(z, top_k=2, top_p=.6), [1, 0, 0, 0]), 'Top-k renormalization precedes top-p'
assert np.allclose(decoding_probs([0, 0, 0], top_k=2), [.5, .5, 0]), 'Stable ID tie-breaking'
assert np.allclose(decoding_probs([10000, 10001]), [1/(1+np.e), np.e/(1+np.e)])
assert np.allclose(decoding_probs(np.log([.2, .8]), temperature=2), [1/3, 2/3])
assert np.allclose(decoding_probs(z, top_p=1), np.exp(z))
assert np.allclose(z, np.log([.5, .3, .15, .05]))
huge = np.array([1e308, -1e308])
before = huge.copy()
with np.errstate(over='raise', under='raise', invalid='raise'):
    expected = np.array([1., np.exp(-2.)])/(1+np.exp(-2.))
    np.testing.assert_allclose(decoding_probs(huge, temperature=1e308, top_p=.9), expected, rtol=1e-14, atol=0.)
    np.testing.assert_allclose(decoding_probs([1e308, 1e308], temperature=.1), [.5, .5], rtol=0., atol=0.)
    np.testing.assert_allclose(decoding_probs([1e308, np.nextafter(1e308, 0.)], temperature=.5), [1., 0.], rtol=0., atol=0.)
    np.testing.assert_allclose(decoding_probs(huge, temperature=.1, top_p=.9), [1., 0.], rtol=0., atol=0.)
    np.testing.assert_allclose(decoding_probs([1e308, 1e308, 1e308], temperature=.25, top_k=2, top_p=.6), [.5, .5, 0.], rtol=0., atol=0.)
    np.testing.assert_allclose(decoding_probs([1e308, 1e308, 1e308], temperature=.25, top_k=2, top_p=.4), [1., 0., 0.], rtol=0., atol=0.)
assert np.array_equal(huge, before)
offset_logits = np.array([2**54, 2**54-2], dtype=float)
before = offset_logits.copy()
expected = np.array([1., np.exp(-2/3)])/(1+np.exp(-2/3))
np.testing.assert_allclose(decoding_probs(offset_logits, temperature=3., top_p=1.), expected, rtol=1e-14, atol=0.)
np.testing.assert_allclose(decoding_probs(offset_logits, temperature=3., top_p=.7), expected, rtol=1e-14, atol=0.)
np.testing.assert_allclose(decoding_probs(offset_logits, temperature=3., top_p=.6), [1., 0.], rtol=0., atol=0.)
assert np.array_equal(offset_logits, before)
base_logits = np.array([-2., 0., -2., -4.])
offset_logits = base_logits + 2**54
before = offset_logits.copy()
for top_k in [None, 1, 2, 3]:
    for top_p in [.6, .7, 1.]:
        expected = _decoding_oracle(base_logits, 3., top_k, top_p)
        np.testing.assert_allclose(decoding_probs(offset_logits, 3., top_k, top_p), expected, rtol=1e-14, atol=0.)
        np.testing.assert_allclose(decoding_probs(offset_logits, 3., top_k, top_p), decoding_probs(base_logits, 3., top_k, top_p), rtol=1e-14, atol=0.)
expected = np.array([np.exp(-2/3), 1., 0., 0.])/(1+np.exp(-2/3))
np.testing.assert_allclose(decoding_probs(offset_logits, temperature=3., top_k=2, top_p=.7), expected, rtol=1e-14, atol=0.)
np.testing.assert_allclose(decoding_probs(offset_logits, temperature=3., top_k=2, top_p=.6), [0., 1., 0., 0.], rtol=0., atol=0.)
assert np.array_equal(offset_logits, before)
rng = np.random.default_rng(717)
for logits in [rng.normal(size=7)*3, np.array([2., 2., -1., 0., 2., -4., .5])]:
    before = logits.copy()
    for temperature in [.1, .8, 1., 3.]:
        for top_k in [None, 1, 3]:
            for top_p in [.4, .9, 1.]:
                expected = _decoding_oracle(logits, temperature, top_k, top_p)
                np.testing.assert_allclose(decoding_probs(logits, temperature, top_k, top_p), expected, rtol=1e-13, atol=1e-15)
    assert np.array_equal(logits, before)
for kwargs in [dict(temperature=0), dict(top_p=0), dict(top_p=1.1), dict(top_k=0), dict(top_k=True), dict(top_k=5)]:
    try:
        decoding_probs(z, **kwargs)
    except ValueError:
        pass
    else:
        raise AssertionError('Reject invalid decoding controls')
print('Decoding-policy checks passed.')


## Existing Course Model

The exact vendored mini_llm configuration and model are embedded here so the notebook does not depend on a repository clone or network import. The Torch model includes learned normalization affine parameters and exact GELU; the browser capstone deliberately omits affine parameters and uses its documented approximation.


In [ ]:
from dataclasses import asdict, dataclass
from typing import Dict, Any


@dataclass
class ModelConfig:
    vocab_size: int
    block_size: int = 64
    n_layer: int = 2
    n_head: int = 4
    n_embd: int = 64
    dropout: float = 0.1
    bias: bool = False

    def __post_init__(self) -> None:
        if self.n_embd % self.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        if self.vocab_size <= 0 or self.block_size <= 0:
            raise ValueError("vocab_size and block_size must be positive")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

"""A bigram baseline and a small decoder-only Transformer."""
import math
from typing import Dict, Optional, Tuple
import torch
import torch.nn as nn
from torch.nn import functional as F

def sample_next_token(logits: torch.Tensor, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
    if temperature <= 0:
        raise ValueError('temperature must be greater than zero')
    logits = logits / temperature
    if top_k is not None:
        top_k = min(top_k, logits.size(-1))
        threshold = torch.topk(logits, top_k).values[:, [-1]]
        logits = logits.masked_fill(logits < threshold, float('-inf'))
    probabilities = F.softmax(logits, dim=-1)
    return torch.multinomial(probabilities, num_samples=1)

class BigramLanguageModel(nn.Module):
    """Each token directly stores logits for the token that follows it."""

    def __init__(self, vocab_size: int) -> None:
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, vocab_size)
        nn.init.zeros_(self.token_embedding.weight)
        self.vocab_size = vocab_size

    def forward(self, inputs: torch.Tensor, targets: Optional[torch.Tensor]=None) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        logits = self.token_embedding(inputs)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), targets.reshape(-1))
        return (logits, loss)

    @torch.no_grad()
    def generate(self, tokens: torch.Tensor, max_new_tokens: int, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
        for _ in range(max_new_tokens):
            logits, _ = self(tokens[:, -1:])
            next_token = sample_next_token(logits[:, -1, :], temperature, top_k)
            tokens = torch.cat((tokens, next_token), dim=1)
        return tokens

class CausalSelfAttention(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.query_key_value = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.projection = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attention_dropout = nn.Dropout(config.dropout)
        self.residual_dropout = nn.Dropout(config.dropout)
        causal_mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer('causal_mask', causal_mask.view(1, 1, config.block_size, config.block_size))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        batch_size, time_steps, channels = inputs.shape
        qkv = self.query_key_value(inputs)
        query, key, value = qkv.split(self.n_embd, dim=2)

        def split_heads(tensor: torch.Tensor) -> torch.Tensor:
            return tensor.view(batch_size, time_steps, self.n_head, self.head_dim).transpose(1, 2)
        query, key, value = map(split_heads, (query, key, value))
        attention = query @ key.transpose(-2, -1) / math.sqrt(self.head_dim)
        attention = attention.masked_fill(self.causal_mask[:, :, :time_steps, :time_steps] == 0, float('-inf'))
        weights = self.attention_dropout(F.softmax(attention, dim=-1))
        output = weights @ value
        output = output.transpose(1, 2).contiguous().view(batch_size, time_steps, channels)
        return self.residual_dropout(self.projection(output))

class FeedForward(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias), nn.GELU(), nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias), nn.Dropout(config.dropout))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.layers(inputs)

class TransformerBlock(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.attention_norm = nn.LayerNorm(config.n_embd)
        self.attention = CausalSelfAttention(config)
        self.feed_forward_norm = nn.LayerNorm(config.n_embd)
        self.feed_forward = FeedForward(config)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        inputs = inputs + self.attention(self.attention_norm(inputs))
        inputs = inputs + self.feed_forward(self.feed_forward_norm(inputs))
        return inputs

class GPT(nn.Module):
    """A compact GPT-2-style decoder-only Transformer."""

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        self.final_norm = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.apply(self._initialize_weights)

    @staticmethod
    def _initialize_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, inputs: torch.Tensor, targets: Optional[torch.Tensor]=None) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        _, time_steps = inputs.shape
        if time_steps > self.config.block_size:
            raise ValueError(f'Sequence length {time_steps} exceeds block size {self.config.block_size}')
        positions = torch.arange(time_steps, device=inputs.device)
        hidden = self.token_embedding(inputs) + self.position_embedding(positions)
        hidden = self.dropout(hidden)
        for block in self.blocks:
            hidden = block(hidden)
        logits = self.lm_head(self.final_norm(hidden))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.config.vocab_size), targets.reshape(-1))
        return (logits, loss)

    def parameter_count(self, exclude_position_embeddings: bool=True) -> int:
        count = sum((parameter.numel() for parameter in self.parameters()))
        if exclude_position_embeddings:
            count -= self.position_embedding.weight.numel()
        return count

    def optimizer_groups(self, weight_decay: float) -> list:
        decay, no_decay = ([], [])
        for parameter in self.parameters():
            (decay if parameter.dim() >= 2 else no_decay).append(parameter)
        return [{'params': decay, 'weight_decay': weight_decay}, {'params': no_decay, 'weight_decay': 0.0}]

    @torch.no_grad()
    def generate(self, tokens: torch.Tensor, max_new_tokens: int, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
        for _ in range(max_new_tokens):
            context = tokens[:, -self.config.block_size:]
            logits, _ = self(context)
            next_token = sample_next_token(logits[:, -1, :], temperature, top_k)
            tokens = torch.cat((tokens, next_token), dim=1)
        return tokens

def build_model(model_type: str, config: ModelConfig) -> nn.Module:
    if model_type == 'bigram':
        return BigramLanguageModel(config.vocab_size)
    if model_type == 'gpt':
        return GPT(config)
    raise ValueError("model_type must be 'bigram' or 'gpt'")

def checkpoint_model(checkpoint: Dict[str, object], device: torch.device) -> nn.Module:
    raw_config = checkpoint.get('model_config')
    model_type = checkpoint.get('model_type')
    if not isinstance(raw_config, dict) or not isinstance(model_type, str):
        raise ValueError('Checkpoint is missing model metadata')
    model = build_model(model_type, ModelConfig(**raw_config))
    model.load_state_dict(checkpoint['model_state'])
    return model.to(device)


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""CPU pretraining, token-weighted accumulation, and exact local resume.

Run: uv run --with numpy --with torch python experiments/scratch_training.py
The notebook embeds the existing mini_llm model source before this cell.
No downloaded data/weights. Fixed synthetic language, not natural-language skill.
"""
import copy
import hashlib
import io
import json
from pathlib import Path
import sys
import torch
from torch.nn import functional as F

if 'GPT' not in globals():
    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'colab/vendor/src'))
    from mini_llm.models import GPT
    from mini_llm.config import ModelConfig


def scratch_data(seed=19):
    """Independent unique sequences: next symbol usually advances by one."""
    generator = torch.Generator().manual_seed(seed)
    rows, seen = [], set()
    while len(rows) < 320:
        row = [int(torch.randint(8, (1,), generator=generator))]
        for _ in range(12):
            step = 1 if torch.rand((), generator=generator) < .82 else 3
            row.append((row[-1] + step) % 8)
        if tuple(row) not in seen:
            seen.add(tuple(row)); rows.append(row)
    data = torch.tensor(rows)
    permutation = torch.randperm(len(data), generator=generator)
    data = data[permutation]
    return data[:224], data[224:272], data[272:]


def scratch_pretrain(seed=7, updates=140):
    torch.set_num_threads(1)
    torch.manual_seed(seed)
    config = ModelConfig(vocab_size=8, block_size=12, n_layer=1, n_head=2, n_embd=24, dropout=0.)
    model = GPT(config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=.006, weight_decay=.01)
    train, validation, test = scratch_data()
    generator = torch.Generator().manual_seed(seed + 1)

    def evaluate(rows):
        model.eval()
        with torch.no_grad():
            logits, _ = model(rows[:, :-1])
            return float(F.cross_entropy(logits.flatten(0, 1), rows[:, 1:].flatten()))

    def next_update(m, opt, gen):
        m.train()
        indices = torch.randint(len(train), (24,), generator=gen)
        batch = train[indices]
        opt.zero_grad(set_to_none=True)
        # Unequal microbatch sizes still implement the full token-average gradient.
        tokens = batch[:, 1:].numel()
        total_loss = 0.
        for micro in (batch[:7], batch[7:]):
            logits, _ = m(micro[:, :-1])
            loss_sum = F.cross_entropy(logits.flatten(0, 1), micro[:, 1:].flatten(), reduction='sum')
            loss = loss_sum / tokens
            loss.backward()
            total_loss += float(loss.detach())
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.)
        opt.step()
        return total_loss

    initial = evaluate(validation)
    history = []
    for step in range(updates):
        next_update(model, optimizer, generator)
        if (step + 1) % 35 == 0:
            history.append({'update': step + 1, 'train_nll': evaluate(train), 'validation_nll': evaluate(validation)})

    # Snapshot immediately before the next sampled update, including RNG state.
    checkpoint = copy.deepcopy(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
        rng=generator.get_state(), torch_rng=torch.get_rng_state(), step=updates, cursor=0,
        config=config.to_dict(), tokenizer_id='symbols-a-through-h-v1'))
    serialized = io.BytesIO()
    torch.save(checkpoint, serialized)
    serialized.seek(0)
    checkpoint = torch.load(serialized, weights_only=True)

    continued = copy.deepcopy(model)
    continued_optimizer = torch.optim.AdamW(continued.parameters(), lr=.006, weight_decay=.01)
    continued_optimizer.load_state_dict(copy.deepcopy(optimizer.state_dict()))
    continued_rng = torch.Generator(); continued_rng.set_state(generator.get_state())
    with torch.random.fork_rng(devices=[]):
        restored = GPT(ModelConfig(**checkpoint['config']))
        restored.load_state_dict(checkpoint['model'])
        resumed_optimizer = torch.optim.AdamW(restored.parameters(), lr=.006, weight_decay=.01)
        resumed_optimizer.load_state_dict(checkpoint['optimizer'])
        resumed_rng = torch.Generator(); resumed_rng.set_state(checkpoint['rng'])
        torch.set_rng_state(checkpoint['torch_rng'])
        uninterrupted = next_update(continued, continued_optimizer, continued_rng)
        torch.set_rng_state(checkpoint['torch_rng'])
        resumed = next_update(restored, resumed_optimizer, resumed_rng)
    assert uninterrupted == resumed
    assert all(torch.equal(a, b) for a, b in zip(continued.parameters(), restored.parameters()))

    # Independent gradient-equivalence check before clipping/optimizer state.
    model.eval()
    batch = train[:9]
    optimizer.zero_grad(set_to_none=True)
    _, full_loss = model(batch[:, :-1], batch[:, 1:]); full_loss.backward()
    full_grads = [p.grad.clone() for p in model.parameters()]
    optimizer.zero_grad(set_to_none=True)
    for micro in (batch[:2], batch[2:]):
        logits, _ = model(micro[:, :-1])
        (F.cross_entropy(logits.flatten(0, 1), micro[:, 1:].flatten(), reduction='sum') / batch[:, 1:].numel()).backward()
    error = max(float((p.grad - g).abs().max()) for p, g in zip(model.parameters(), full_grads))
    assert error < 2e-6, error
    final = evaluate(validation)
    assert final < initial, 'This fixed toy run should learn its transition pattern'
    model.eval(); torch.manual_seed(seed + 100)
    sample = model.generate(torch.tensor([[0]]), 16, top_k=3).tolist()[0]
    report = dict(seed=seed, updates=updates, config=config.to_dict(), parameters=model.parameter_count(False),
        split_sizes=[len(train), len(validation), len(test)],
        data_sha256=hashlib.sha256(torch.cat([train, validation, test]).numpy().tobytes()).hexdigest(),
        initial_validation_nll=initial, final_validation_nll=final, test_nll=evaluate(test),
        accumulation_max_error=error, resume_exact=True, history=history, generated_ids=sample,
        tokenizer=dict(symbols=list('abcdefgh'),ids=list(range(8))),generated_text=''.join('abcdefgh'[i] for i in sample),
        limitation='Synthetic transition language; not a general-purpose LLM. Test used once after a fixed recipe.')
    return model, report


if __name__ == '__main__':
    _, report = scratch_pretrain()
    print(json.dumps(report, indent=2))
